In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

In [2]:
#Pydantic

from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="The movie released in which year")
    Directot:str = Field(description="The director of the movie")
    rating:float = Field(description="The rating of the movie")

In [3]:
model_with_structure = model.with_structured_output(Movie)

In [4]:
model.invoke("Tell me about the movie TENET")

AIMessage(content='<think>\nOkay, the user is asking about the movie Tenet. Let me start by recalling what I know about it. Directed by Christopher Nolan, right? It\'s a 2020 action film. The main character is a protagonist without a name, often referred to as the Protagonist. There\'s a guy named Kenneth Branagh in it. The plot involves time inversion, which is a concept where objects and people move backward through time. \n\nThe user might want a summary of the movie, so I should outline the main storyline. The Protagonist is part of a secret organization called Tenet, which deals with these inverted objects. The mission involves preventing a war by stopping a future arms dealer named Sator from causing a catastrophic event. There\'s a lot of action sequences, like the car chase in Tallinn, and the inversion of the SUV. \n\nThemes to mention could include time, entropy, and the idea of reversing time. The movie\'s structure is non-linear, which is typical for Nolan, with the climax 

In [5]:
response = model_with_structure.invoke("Tell me about the movie TENET")
response

Movie(title='TENET', year=2020, Directot='Christopher Nolan', rating=8.0)

In [6]:
#Pydantic

from pydantic import BaseModel,Field
'''Details of the movie'''
class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="The movie released in which year")
    Directot:str = Field(description="The director of the movie")
    rating:float = Field(description="The rating of the movie")
    
model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("provide the details of the movie Bahubali")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Bahubali. Let me check what tools I have available. There's a Movie function that requires title, year, director, and rating. I need to make sure I have all that information.\n\nFirst, the title is clearly Bahubali. The original Bahubali movie was released in 2015. The director is S.S. Rajamouli. As for the rating, I think it's around 8.1 on IMDb. Let me confirm those details. Yeah, that's correct. So I'll structure the tool call with those parameters. Make sure the JSON is properly formatted with the required fields. No need to mention other parts, just the tool call as specified.\n", 'tool_calls': [{'id': 'sm33qzgj6', 'function': {'arguments': '{"Directot":"S.S. Rajamouli","rating":8.1,"title":"Bahubali","year":2015}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 208, 'prompt_tokens': 226, 'total_tokens': 434, 

In [7]:
##Nested Output

from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str
    
class Movie_Details(BaseModel):
    title:str = Field(...,description="The title of the movie")
    actor:list[Actor] = Field(...,description="The Actors and roles of the movie")
    genre:list[str] = Field(...,description="The genre of the movie")
    budget:int = Field(...,description="The budget of the movie in INR")
    
model_with_structure = model.with_structured_output(Movie_Details)

response = model_with_structure.invoke("provide the details of the movie Bahubali")
response

Movie_Details(title='Bahubali', actor=[Actor(name='Prabhas', role='Bahubali'), Actor(name='Tamannaah Bhatia', role='Devasena'), Actor(name='Sudeep', role='Bhallala Deva')], genre=['Action', 'Adventure', 'Fantasy'], budget=2500000000)

In [9]:
##Typedict

from typing_extensions import TypedDict,Annotated

class Moviedetails(TypedDict):
    title:Annotated[str,...,"the title of the Movie"]
    year:Annotated[int,...,"The year of movie released"]
    Director:Annotated[str,...,"The director of the movie"]
    rating :Annotated[str,...,"the movie year released"]
    
model_with_typeddict = model.with_structured_output(Moviedetails)

response = model_with_typeddict.invoke("Tell me about the movie tenet")
response

{'Director': 'Christopher Nolan',
 'rating': '7.5/10',
 'title': 'Tenet',
 'year': 2020}

In [ ]:
##Nested Output

class Actor(TypedDict):
    name:str
    role:str
    
class Movie_Details(TypedDict):
    title:str = Field(...,description="The title of the movie")
    actor:list[Actor] = Field(...,description="The Actors and roles of the movie")
    genre:list[str] = Field(...,description="The genre of the movie")
    budget:int = Field(...,description="The budget of the movie in INR")
    
model_with_structure = model.with_structured_output(Movie_Details)

response = model_with_structure.invoke("provide the details of the movie Bahubali")
response

{'actor': [{'name': 'Prabhas', 'role': 'Bahubali'},
  {'name': 'Tamannaah Bhatia', 'role': 'Devasena'},
  {'name': 'Sushruta', 'role': 'Bhallaladeva'}],
 'budget': 250000000,
 'genre': ['Action', 'Mythology', 'Drama'],
 'title': 'Bahubali'}

In [11]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}